In [ ]:
!pip install transformers datasets accelerate
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
print("HF_TOKEN loaded from Colab secrets and set as environment variable.")

HF_TOKEN loaded from Colab secrets and set as environment variable.


## Phase 1

In [ ]:
# ============================================================
# Phase 1: CinePile Text-Only Evaluation — Final Fix
# ============================================================

import gc
import torch
import pandas as pd
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModel,
    BitsAndBytesConfig
)
from tqdm import tqdm


# ============================================================
# 1. Configure Class
# ============================================================
@dataclass
class ExperimentConfig:
    dataset_name: str           = 'tomg-group-umd/cinepile'
    dataset_split: str          = 'test'
    max_samples: Optional[int]  = None

    max_scene_length: int       = 1500
    max_input_tokens: int       = 2048
    deberta_max_tokens: int     = 512

    max_new_tokens: int         = 5
    do_sample: bool             = False

    load_in_4bit: bool          = True
    bnb_4bit_quant_type: str    = "nf4"
    bnb_4bit_compute_dtype: torch.dtype = torch.bfloat16

    models: list = field(default_factory=lambda: [
        ('DeBERTa-v3-base', 'microsoft/deberta-v3-base',          'deberta'),
        ('Llama-3.1-8B',    'meta-llama/Llama-3.1-8B-Instruct',  'llm'),
        ('Mistral-7B',      'mistralai/Mistral-7B-Instruct-v0.3', 'llm'),
    ])

    output_csv: str = 'phase1_results.csv'

    category_map: dict = field(default_factory=lambda: {
        'TEMP': 'Temporal',
        'CRD':  'Character and',
        'NPA':  'Narrative and',
        'STA':  'Setting and',
        'TH':   'Theme Exploration',
    })

# ============================================================
# 2. Prepare dataset
# ============================================================
class CinePileDataset:

    LETTERS = ['A', 'B', 'C', 'D', 'E']

    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.data   = self._load()

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:


        for i, choice in enumerate(choices):
            if answer_text == choice:
                return cls.LETTERS[i]


        answer_stripped = answer_text.strip()
        for i, choice in enumerate(choices):
            if answer_stripped == choice.strip():
                return cls.LETTERS[i]

        for i, choice in enumerate(choices):
            if answer_stripped in choice or choice.strip() in answer_stripped:
                return cls.LETTERS[i]

        print(f"  [Warning] Cannot match answer_key to choices.")
        print(f"    answer_key : {repr(answer_text)}")
        print(f"    choices    : {choices}")
        return 'A'  # fallback

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == 'true'

    def _load(self):
        raw = load_dataset(
            self.config.dataset_name,
            split=self.config.dataset_split
        )
        samples = []
        unmatched = 0

        for ex in raw:
            choices    = ex['choices']
            answer_key = ex['answer_key']

            # answer_key
            letter = self._answer_text_to_letter(answer_key, choices)
            if letter == 'A' and answer_key != choices[0]:
                unmatched += 1

            samples.append({
                'movie_scene':       ex['movie_scene'],
                'question':          ex['question'],
                'choices':           choices,
                'answer_key':        letter,           # 'A'~'E'
                'answer_text':       answer_key,
                'question_category': ex['question_category'],
                'hard_split':        self._normalize_hard_split(ex['hard_split']),
            })

        if self.config.max_samples:
            samples = samples[:self.config.max_samples]

        print(f"[Dataset] Loaded {len(samples)} samples")
        print(f"          answer_key sample : "
              f"{repr(samples[0]['answer_text'])} → {samples[0]['answer_key']}")
        print(f"          hard_split sample : {samples[0]['hard_split']}")
        if unmatched > 0:
            print(f"  [Warning] {unmatched} samples fell back to 'A' "
                  f"(answer not matched in choices)")
        return samples

    def run_sanity_check(self, n: int = 5):
        print("\n=== Sanity Check ===")
        for i, s in enumerate(self.data[:n]):
            idx = ord(s['answer_key']) - 65
            matched_text = s['choices'][idx]
            match_ok = matched_text.strip() == s['answer_text'].strip()
            print(f"  [{i}] {s['answer_key']} → \"{matched_text}\"")
            print(f"       original: \"{s['answer_text']}\"  match={match_ok}")
        print()


# ============================================================
# 3. Base Evaluator
# ============================================================
class BaseEvaluator(ABC):
    def __init__(self, model_id: str, config: ExperimentConfig):
        self.model_id  = model_id
        self.config    = config
        self.model     = None
        self.tokenizer = None

    @abstractmethod
    def load_model(self): pass

    @abstractmethod
    def predict(self, sample: dict) -> str: pass

    def evaluate(self, data: list) -> pd.DataFrame:
        records = []
        for sample in tqdm(data, desc=f"  {self.model_id}"):
            pred = self.predict(sample)
            records.append({
                'pred':              pred,
                'label':             sample['answer_key'],
                'question_category': sample['question_category'],
                'hard_split':        sample['hard_split'],
            })
        return pd.DataFrame(records)

    def free_memory(self):
        if self.model is not None:
            del self.model
            self.model = None
        if self.tokenizer is not None:
            del self.tokenizer
            self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  [Memory freed] {self.model_id}")


# ============================================================
# 4. LLM Evaluator
# ============================================================
class LLMEvaluator(BaseEvaluator):
    def load_model(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = self.config.load_in_4bit,
            bnb_4bit_use_double_quant = True,
            bnb_4bit_quant_type       = self.config.bnb_4bit_quant_type,
            bnb_4bit_compute_dtype    = self.config.bnb_4bit_compute_dtype,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        self.model.eval()
        print(f"  [Loaded] {self.model_id} (4-bit)")

    def _build_prompt(self, sample: dict) -> str:
        scene   = sample['movie_scene'][:self.config.max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            "Answer with only the letter (A, B, C, D, or E):"
        )

    def predict(self, sample: dict) -> str:
        prompt = self._build_prompt(sample)
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.config.max_input_tokens,
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.config.max_new_tokens,
                do_sample=self.config.do_sample,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        decoded = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        for char in decoded:
            if char in 'ABCDE':
                return char
        return 'A'  # fallback


# ============================================================
# 5. DeBERTa-v3 Evaluator
# ============================================================
class DeBERTaEvaluator(BaseEvaluator):

    def load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model     = AutoModel.from_pretrained(self.model_id).to(
            'cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.model.eval()
        print(f"  [Loaded] {self.model_id}")

    def _score(self, text_a: str, text_b: str) -> float:
        inputs = self.tokenizer(
            text_a, text_b,
            return_tensors='pt',
            truncation=True,
            max_length=self.config.deberta_max_tokens,
            padding=True,
        ).to(self.model.device)
        with torch.no_grad():
            cls_emb = self.model(**inputs).last_hidden_state[:, 0, :]
        return cls_emb.norm(dim=-1).item()

    def predict(self, sample: dict) -> str:
        scene_q = (
            f"{sample['question']} "
            f"{sample['movie_scene'][:self.config.deberta_max_tokens // 2]}"
        )
        scores = [self._score(scene_q, c) for c in sample['choices']]
        return chr(65 + scores.index(max(scores))) # 'A'~'E'


# ============================================================
# 6. Metrics calculation
# ============================================================
class MetricsCalculator:
    def __init__(self, config: ExperimentConfig):
        self.config = config

    def compute(self, df: pd.DataFrame) -> dict:
        metrics = {}
        correct = df['pred'] == df['label']

        metrics['Overall']   = correct.mean()
        metrics['Overall_n'] = len(df)

        for abbr, cat_name in self.config.category_map.items():
            mask = df['question_category'].str.contains(
                cat_name, case=False, na=False
            )
            n = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f'{abbr}_n'] = n

        hard_mask = df['hard_split'] == True
        n_hard    = int(hard_mask.sum())
        metrics['Hard']   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics['Hard_n'] = n_hard

        return metrics


# ============================================================
# 7. Phase1 Runner
# ============================================================
class Phase1Runner:
    EVALUATOR_REGISTRY = {
        'llm':     LLMEvaluator,
        'deberta': DeBERTaEvaluator,
    }

    def __init__(self, config: ExperimentConfig):
        self.config       = config
        self.dataset      = CinePileDataset(config)
        self.metrics_calc = MetricsCalculator(config)
        self.all_metrics  = {}

        # 运行 sanity check，确认 answer_key 转换正确
        self.dataset.run_sanity_check(n=3)

    def run(self):
        print("="*60)
        print("PHASE 1: CinePile Text-Only Evaluation")
        print("="*60)

        for display_name, model_id, eval_type in self.config.models:
            print(f"\n>>> [{eval_type.upper()}] {display_name}")

            evaluator_cls = self.EVALUATOR_REGISTRY[eval_type]
            evaluator     = evaluator_cls(model_id, self.config)
            evaluator.load_model()

            results_df = evaluator.evaluate(self.dataset.data)

            metrics = self.metrics_calc.compute(results_df)
            self.all_metrics[display_name] = metrics

            hard_str = (f"{metrics['Hard']:.2%}"
                        if metrics.get('Hard') is not None else "N/A")
            print(f"  Overall: {metrics['Overall']:.2%}  "
                  f"| Hard: {hard_str}  "
                  f"| n={metrics['Overall_n']}")

            # Free memory after each model run
            evaluator.free_memory()

        self._display_and_save()

    def _display_and_save(self):
        display_cols = ['Overall', 'TEMP', 'CRD', 'NPA', 'STA', 'TH', 'Hard']
        rows = []
        for model_name, metrics in self.all_metrics.items():
            row = {'Model': model_name}
            for col in display_cols:
                val = metrics.get(col)
                n   = metrics.get(f'{col}_n', metrics.get('Overall_n', ''))
                row[col] = f"{val:.1%}(n={n})" if val is not None else 'N/A'
            rows.append(row)

        df = pd.DataFrame(rows)
        print("\n" + "="*60)
        print("FINAL RESULTS — CinePile Text-Only")
        print("="*60)
        print(df.to_string(index=False))
        df.to_csv(self.config.output_csv, index=False)
        print(f"\nSaved → {self.config.output_csv}")


if __name__ == '__main__':

    # config = ExperimentConfig(
    #     max_samples     = None,     # using None for all test datasets
    #     max_scene_length= 1500,
    #     load_in_4bit    = True,
    #     output_csv      = 'phase1_mistral_results.csv',
    #     models=[
    #         # ('DeBERTa-v3-base', 'microsoft/deberta-v3-base',          'deberta'),
    #         # ('Llama-3.1-8B',    'meta-llama/Llama-3.1-8B-Instruct',  'llm'),
    #         ('Mistral-7B',      'mistralai/Mistral-7B-Instruct-v0.3', 'llm'),
    #     ]
    # )

    config = ExperimentConfig(
        max_samples     = None,     # using None for all test datasets
        max_scene_length= 1500,
        load_in_4bit    = True,
        output_csv      = 'phase1_llama_results.csv',
        models=[
            ('DeBERTa-v3-base', 'microsoft/deberta-v3-base',          'deberta'),
            # ('Llama-3.1-8B',    'meta-llama/Llama-3.1-8B-Instruct',  'llm'),
            # ('Mistral-7B',      'mistralai/Mistral-7B-Instruct-v0.3', 'llm'),
        ]
    )

    runner = Phase1Runner(config)
    runner.run()

[Dataset] Loaded 4941 samples
          answer_key sample : 'From anxiety to excitement' → E
          hard_split sample : True

=== Sanity Check ===
  [0] E → "From anxiety to excitement"
       original: "From anxiety to excitement"  match=True
  [1] E → "Suggests next steps"
       original: "Suggests next steps"  match=True
  [2] D → "The rough foliage"
       original: "The rough foliage"  match=True

PHASE 1: CinePile Text-Only Evaluation

>>> [DEBERTA] DeBERTa-v3-base


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [Loaded] microsoft/deberta-v3-base


  microsoft/deberta-v3-base: 100%|██████████| 4941/4941 [11:20<00:00,  7.26it/s]


  Overall: 21.72%  | Hard: 20.63%  | n=4941
  [Memory freed] microsoft/deberta-v3-base

FINAL RESULTS — CinePile Text-Only
          Model       Overall         TEMP           CRD          NPA           STA           TH         Hard
DeBERTa-v3-base 21.7%(n=4941) 20.8%(n=688) 21.9%(n=2068) 20.3%(n=463) 22.8%(n=1532) 17.9%(n=190) 20.6%(n=887)

Saved → phase1_llama_results.csv


In [ ]:
##  phase 1 results
'''
============================================================
FINAL RESULTS — CinePile Text-Only
============================================================
          Model       Overall         TEMP           CRD          NPA           STA           TH         Hard
DeBERTa-v3-base 21.7%(n=4941) 20.8%(n=688) 21.9%(n=2068) 20.3%(n=463) 22.8%(n=1532) 17.9%(n=190) 20.6%(n=887)

     Model       Overall         TEMP           CRD          NPA           STA           TH         Hard
Mistral-7B 55.2%(n=4941) 34.9%(n=688) 57.4%(n=2068) 55.7%(n=463) 61.6%(n=1532) 52.6%(n=190) 32.7%(n=887)


Model	           Overall	       TEMP	          CRD	         NPA	         STA	         TH	        Hard
Llama-3.1-8B	60.0%(n=4941)	40.3%(n=688)	62.3%(n=2068)	56.4%(n=463)	67.8%(n=1532)	53.2%(n=190)	38.8%(n=887)
'''

## Phase 2

In [ ]:
!pip install transformers datasets accelerate bitsandbytes peft trl pyreft

In [ ]:
from google.colab import userdata
import os

# Retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN loaded from Colab secrets and set as environment variable.")
# Now, when load_dataset is called, it will automatically use this token for authentication.

HF_TOKEN loaded from Colab secrets and set as environment variable.


In [ ]:
# ============================================================
# Phase 2: PEFT Fine-tuning on CinePile (Text-Only)
# Methods: QLoRA, Prefix Tuning, ReFT (LoReFT)
# !pip install transformers datasets accelerate bitsandbytes peft trl pyreft
# ============================================================

import gc
import os
import torch
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
)
from peft import (
    LoraConfig,
    PrefixTuningConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm


# ============================================================
# 1. Config
# ============================================================
@dataclass
class Phase2Config:
    # --- dataset ---
    dataset_name: str          = 'tomg-group-umd/cinepile'
    train_split: str           = 'train'
    test_split: str            = 'test'
    max_train_samples: Optional[int] = 2000
    max_test_samples: Optional[int]  = 500

    # --- base model---
    base_model_id: str         = 'meta-llama/Llama-3.1-8B-Instruct'

    # --- input ---
    max_scene_length: int      = 1000   #
    max_seq_length: int        = 1024   #

    # --- parameter for training ---
    num_train_epochs: int      = 2
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4   #  batch size = 8
    learning_rate: float       = 2e-4
    warmup_ratio: float        = 0.05
    lr_scheduler_type: str     = 'cosine'
    fp16: bool                 = False
    bf16: bool                 = True
    logging_steps: int         = 20
    save_steps: int            = 200
    output_base_dir: str       = './phase2_outputs'

    # --- QLoRA param ---
    lora_r: int                = 16
    lora_alpha: int            = 32
    lora_dropout: float        = 0.05
    lora_target_modules: list  = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ])

    # --- Prefix Tuning param ---
    prefix_num_virtual_tokens: int = 20

    # --- ReFT (LoReFT) param ---
    reft_rank: int             = 4
    reft_layers: list          = field(default_factory=lambda: [8, 16, 24])

    # --- evaluation ---
    max_new_tokens: int        = 5
    output_csv: str            = 'phase2_results.csv'

    # --- mapping: the same as Phase 1---
    category_map: dict = field(default_factory=lambda: {
        'TEMP': 'Temporal',
        'CRD':  'Character and',
        'NPA':  'Narrative and',
        'STA':  'Setting and',
        'TH':   'Theme Exploration',
    })


# ============================================================
# 2. Dataset
# ============================================================
class CinePileDataset:
    LETTERS = ['A', 'B', 'C', 'D', 'E']

    def __init__(self, config: Phase2Config):
        self.config = config
        self.train_data = self._load(config.train_split, config.max_train_samples)
        self.test_data  = self._load(config.test_split,  config.max_test_samples)
        print(f"[Dataset] Train: {len(self.train_data)} | Test: {len(self.test_data)}")

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        for i, c in enumerate(choices):
            if answer_text.strip() == c.strip():
                return cls.LETTERS[i]
        for i, c in enumerate(choices):
            if answer_text.strip() in c or c.strip() in answer_text:
                return cls.LETTERS[i]
        return 'A'

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == 'true'

    def _load(self, split: str, max_samples: Optional[int]) -> list:
        raw = load_dataset(self.config.dataset_name, split=split)
        samples = []
        for ex in raw:
            letter = self._answer_text_to_letter(ex['answer_key'], ex['choices'])
            samples.append({
                'movie_scene':       ex['movie_scene'],
                'question':          ex['question'],
                'choices':           ex['choices'],
                'answer_key':        letter,
                'question_category': ex['question_category'],
                'hard_split':        self._normalize_hard_split(ex['hard_split']),
            })
        if max_samples:
            samples = samples[:max_samples]
        return samples


# ============================================================
# 3. Prompt
# ============================================================
class PromptBuilder:

    @staticmethod
    def build_train_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample['movie_scene'][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
        )
        answer  = sample['answer_key']
        return (
            f"You are watching a movie. Based on the scene description, "
            f"answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            f"Answer: {answer}"   # with answer
        )

    @staticmethod
    def build_inference_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample['movie_scene'][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
        )
        return (
            f"You are watching a movie. Based on the scene description, "
            f"answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            f"Answer:"   # without answer
        )


# ============================================================
# 4. Evaluator
# ============================================================
class Evaluator:
    def __init__(self, config: Phase2Config):
        self.config = config

    def evaluate(self, model, tokenizer, test_data: list,
                 method_name: str) -> dict:
        model.eval()
        records = []
        for sample in tqdm(test_data, desc=f"  Evaluating {method_name}"):
            pred = self._predict(model, tokenizer, sample)
            records.append({
                'pred':              pred,
                'label':             sample['answer_key'],
                'question_category': sample['question_category'],
                'hard_split':        sample['hard_split'],
            })
        df = pd.DataFrame(records)
        return self._compute_metrics(df)

    def _predict(self, model, tokenizer, sample: dict) -> str:
        prompt  = PromptBuilder.build_inference_prompt(
            sample, self.config.max_scene_length
        )
        inputs  = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.config.max_seq_length,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.config.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        for char in decoded:
            if char in 'ABCDE':
                return char
        return 'A'

    def _compute_metrics(self, df: pd.DataFrame) -> dict:
        correct  = df['pred'] == df['label']
        metrics  = {
            'Overall':   correct.mean(),
            'Overall_n': len(df),
        }
        for abbr, keyword in self.config.category_map.items():
            mask = df['question_category'].str.contains(
                keyword, case=False, na=False
            )
            n = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f'{abbr}_n'] = n

        hard_mask    = df['hard_split'] == True
        n_hard       = int(hard_mask.sum())
        metrics['Hard']   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics['Hard_n'] = n_hard
        return metrics


# ============================================================
# 5. Base Trainer
# ============================================================
class BasePEFTTrainer(ABC):
    def __init__(self, config: Phase2Config):
        self.config    = config
        self.model     = None
        self.tokenizer = None
        self.method_name = "base"

    def _load_base_model_4bit(self):
        """加载 4-bit 量化基础模型（QLoRA 用）"""
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_use_double_quant = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = torch.bfloat16,
        )
        tokenizer = AutoTokenizer.from_pretrained(self.config.base_model_id)
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = AutoModelForCausalLM.from_pretrained(
            self.config.base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
        return model, tokenizer

    def _load_base_model_bf16(self):
        """加载 bf16 基础模型（Prefix Tuning / ReFT 用）"""
        tokenizer = AutoTokenizer.from_pretrained(self.config.base_model_id)
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = AutoModelForCausalLM.from_pretrained(
            self.config.base_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        return model, tokenizer

    def _make_hf_dataset(self, data: list):
        """将样本列表转为 HuggingFace Dataset（SFTTrainer 需要）"""
        from datasets import Dataset
        return Dataset.from_list([
            {'text': PromptBuilder.build_train_prompt(
                s, self.config.max_scene_length
            )}
            for s in data
        ])

    def _output_dir(self) -> str:
        path = os.path.join(self.config.output_base_dir, self.method_name)
        os.makedirs(path, exist_ok=True)
        return path

    def free_memory(self):
        if self.model is not None:
            del self.model
            self.model = None
        if self.tokenizer is not None:
            del self.tokenizer
            self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  [Memory freed] {self.method_name}")

    @abstractmethod
    def train(self, train_data: list): pass

    @abstractmethod
    def load_for_inference(self): pass


# ============================================================
# 6. QLoRA Trainer
# ============================================================
class QLoRATrainer(BasePEFTTrainer):

    def __init__(self, config: Phase2Config):
        super().__init__(config)
        self.method_name = f"qlora_r{config.lora_r}"

    def train(self, train_data: list):
        print(f"\n[QLoRA] Loading base model (4-bit)...")
        model, tokenizer = self._load_base_model_4bit()

        lora_config = LoraConfig(
            r                = self.config.lora_r,
            lora_alpha       = self.config.lora_alpha,
            lora_dropout     = self.config.lora_dropout,
            target_modules   = self.config.lora_target_modules,
            bias             = "none",
            task_type        = TaskType.CAUSAL_LM,
        )

        train_args = SFTConfig(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.config.num_train_epochs,
            per_device_train_batch_size = self.config.per_device_train_batch_size,
            gradient_accumulation_steps = self.config.gradient_accumulation_steps,
            learning_rate               = self.config.learning_rate,
            warmup_ratio                = self.config.warmup_ratio,
            lr_scheduler_type           = self.config.lr_scheduler_type,
            bf16                        = self.config.bf16,
            logging_steps               = self.config.logging_steps,
            save_steps                  = self.config.save_steps,
            max_seq_length              = self.config.max_seq_length,
            dataset_text_field          = "text",
            report_to                   = "none",
            gradient_checkpointing      = True,
            optim                       = "paged_adamw_8bit",
        )

        hf_dataset = self._make_hf_dataset(train_data)

        trainer = SFTTrainer(
            model         = model,
            args          = train_args,
            train_dataset = hf_dataset,
            peft_config   = lora_config,
            tokenizer     = tokenizer,
        )

        print(f"[QLoRA] Training... (samples={len(train_data)})")
        trainer.train()
        trainer.save_model(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[QLoRA] Saved → {self._output_dir()}")

        self.model     = trainer.model
        self.tokenizer = tokenizer

    def load_for_inference(self):
        """从保存的 checkpoint 加载用于推理"""
        print(f"[QLoRA] Loading from {self._output_dir()}")
        model, tokenizer = self._load_base_model_4bit()
        self.model     = PeftModel.from_pretrained(model, self._output_dir())
        self.tokenizer = tokenizer


# ============================================================
# 7. Prefix Tuning Trainer
# ============================================================
class PrefixTuningTrainer(BasePEFTTrainer):

    def __init__(self, config: Phase2Config):
        super().__init__(config)
        self.method_name = f"prefix_vt{config.prefix_num_virtual_tokens}"

    def train(self, train_data: list):
        print(f"\n[Prefix] Loading base model (bf16)...")
        model, tokenizer = self._load_base_model_bf16()

        prefix_config = PrefixTuningConfig(
            task_type          = TaskType.CAUSAL_LM,
            num_virtual_tokens = self.config.prefix_num_virtual_tokens,
            prefix_projection  = True,
        )

        model = get_peft_model(model, prefix_config)
        model.print_trainable_parameters()

        train_args = SFTConfig(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.config.num_train_epochs,
            per_device_train_batch_size = self.config.per_device_train_batch_size,
            gradient_accumulation_steps = self.config.gradient_accumulation_steps,
            learning_rate               = 5e-4,   # Prefix Tuning 需要更大 lr
            warmup_ratio                = self.config.warmup_ratio,
            lr_scheduler_type           = self.config.lr_scheduler_type,
            bf16                        = self.config.bf16,
            logging_steps               = self.config.logging_steps,
            save_steps                  = self.config.save_steps,
            max_seq_length              = self.config.max_seq_length,
            dataset_text_field          = "text",
            report_to                   = "none",
            gradient_checkpointing      = False,  # Prefix Tuning 不兼容 grad ckpt
        )

        hf_dataset = self._make_hf_dataset(train_data)

        trainer = SFTTrainer(
            model         = model,
            args          = train_args,
            train_dataset = hf_dataset,
            tokenizer     = tokenizer,
        )

        print(f"[Prefix] Training... (samples={len(train_data)})")
        trainer.train()
        trainer.save_model(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[Prefix] Saved → {self._output_dir()}")

        self.model     = model
        self.tokenizer = tokenizer

    def load_for_inference(self):
        print(f"[Prefix] Loading from {self._output_dir()}")
        model, tokenizer = self._load_base_model_bf16()
        self.model     = PeftModel.from_pretrained(model, self._output_dir())
        self.tokenizer = tokenizer


# ============================================================
# 8. ReFT (LoReFT) Trainer
# ============================================================
class ReFTTrainer(BasePEFTTrainer):

    def __init__(self, config: Phase2Config):
        super().__init__(config)
        self.method_name = f"reft_r{config.reft_rank}"

    def train(self, train_data: list):
        try:
            import pyreft
        except ImportError:
            print("[ReFT] pyreft not installed. Run: pip install pyreft")
            return

        print(f"\n[ReFT] Loading base model (bf16)...")
        model, tokenizer = self._load_base_model_bf16()

        reft_config = pyreft.ReftConfig(
            representations=[
                {
                    "layer"           : l,
                    "component"       : "block_output",
                    "low_rank_dimension": self.config.reft_rank,
                    "intervention"    : pyreft.LoreftIntervention(
                        embed_dim          = model.config.hidden_size,
                        low_rank_dimension = self.config.reft_rank,
                    )
                }
                for l in self.config.reft_layers
            ]
        )
        reft_model = pyreft.get_reft_model(model, reft_config)
        reft_model.print_trainable_parameters()

        train_dataset = self._make_reft_dataset(
            train_data, tokenizer, pyreft
        )

        training_args = TrainingArguments(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.config.num_train_epochs,
            per_device_train_batch_size = self.config.per_device_train_batch_size,
            gradient_accumulation_steps = self.config.gradient_accumulation_steps,
            learning_rate               = self.config.learning_rate,
            warmup_ratio                = self.config.warmup_ratio,
            bf16                        = self.config.bf16,
            logging_steps               = self.config.logging_steps,
            save_steps                  = self.config.save_steps,
            report_to                   = "none",
        )

        reft_trainer = pyreft.ReftTrainerForCausalLM(
            model         = reft_model,
            args          = training_args,
            train_dataset = train_dataset,
            tokenizer     = tokenizer,
            data_collator = pyreft.ReftDataCollator(
                data_collator=DataCollatorWithPadding(tokenizer)
            ),
        )

        print(f"[ReFT] Training... (samples={len(train_data)})")
        reft_trainer.train()
        reft_model.save(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[ReFT] Saved → {self._output_dir()}")

        self.model     = reft_model
        self.tokenizer = tokenizer

    def _make_reft_dataset(self, data, tokenizer, pyreft):

        from datasets import Dataset
        prompts, outputs = [], []
        for s in data:
            prompt = PromptBuilder.build_inference_prompt(
                s, self.config.max_scene_length
            )
            prompts.append(prompt)
            outputs.append(s['answer_key'])

        return pyreft.make_last_position_supervised_data_module(
            tokenizer     = tokenizer,
            model         = self.model,
            inputs        = prompts,
            outputs       = outputs,
            num_interventions = len(self.config.reft_layers),
        )

    def load_for_inference(self):
        try:
            import pyreft
        except ImportError:
            print("[ReFT] pyreft not installed.")
            return

        print(f"[ReFT] Loading from {self._output_dir()}")
        model, tokenizer = self._load_base_model_bf16()
        self.model     = pyreft.ReftModel.load(self._output_dir(), model)
        self.tokenizer = tokenizer


# ============================================================
# 9. Phase2
# ============================================================
class Phase2Runner:
    def __init__(self, config: Phase2Config):
        self.config      = config
        self.dataset     = CinePileDataset(config)
        self.evaluator   = Evaluator(config)
        self.all_metrics = {}


        self.trainers = [
            # QLoRATrainer(config),
            PrefixTuningTrainer(config),
            # ReFTTrainer(config),
        ]

    def run(self):
        print("\n" + "="*60)
        print("PHASE 2: PEFT Fine-tuning on CinePile")
        print(f"Base model : {self.config.base_model_id}")
        print(f"Train size : {len(self.dataset.train_data)}")
        print(f"Test size  : {len(self.dataset.test_data)}")
        print("="*60)

        for trainer in self.trainers:
            print(f"\n{'='*60}")
            print(f"Method: {trainer.method_name.upper()}")
            print(f"{'='*60}")


            trainer.train(self.dataset.train_data)


            if trainer.model is not None:
                metrics = self.evaluator.evaluate(
                    trainer.model,
                    trainer.tokenizer,
                    self.dataset.test_data,
                    trainer.method_name,
                )
                self.all_metrics[trainer.method_name] = metrics
                hard_str = (f"{metrics['Hard']:.2%}"
                            if metrics.get('Hard') is not None else "N/A")
                print(f"  Overall: {metrics['Overall']:.2%} "
                      f"| Hard: {hard_str}")


            trainer.free_memory()

        self._display_and_save()

    def _display_and_save(self):
        display_cols = ['Overall', 'TEMP', 'CRD', 'NPA', 'STA', 'TH', 'Hard']
        rows = []
        for method_name, metrics in self.all_metrics.items():
            row = {'Method': method_name}
            for col in display_cols:
                val = metrics.get(col)
                n   = metrics.get(f'{col}_n', metrics.get('Overall_n', ''))
                row[col] = f"{val:.1%}(n={n})" if val is not None else 'N/A'
            rows.append(row)

        df = pd.DataFrame(rows)
        print("\n" + "="*60)
        print("PHASE 2 FINAL RESULTS — CinePile PEFT Fine-tuning")
        print("="*60)
        print(df.to_string(index=False))
        df.to_csv(self.config.output_csv, index=False)
        print(f"\nSaved → {self.config.output_csv}")


# ============================================================
# 10. Main
# ============================================================
if __name__ == '__main__':

    config = Phase2Config(
        base_model_id        = 'meta-llama/Llama-3.1-8B-Instruct',
        max_train_samples    = 20,
        max_test_samples     = 500,
        num_train_epochs     = 2,
        lora_r               = 16,
        prefix_num_virtual_tokens = 20,
        reft_rank            = 4,
        reft_layers          = [8, 16, 24],
        output_base_dir      = './phase2_outputs',
        output_csv           = 'phase2_results.csv',
    )

    runner = Phase2Runner(config)
    runner.run()


README.md: 0.00B [00:00, ?B/s]

v2/train-00000-of-00003.parquet:   0%|          | 0.00/21.9M [00:00<?, ?B/s]

v2/train-00001-of-00003.parquet:   0%|          | 0.00/23.2M [00:00<?, ?B/s]

v2/train-00002-of-00003.parquet:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

v2/test-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

TypeError: 'LoreftIntervention' object is not subscriptable

### 2.3 Prefix Tuning

In [ ]:
!pip install transformers datasets accelerate
!pip install -U bitsandbytes>=0.46.1
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from google.colab import userdata
import os

# Retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN loaded from Colab secrets and set as environment variable.")
# Now, when load_dataset is called, it will automatically use this token for authentication.

HF_TOKEN loaded from Colab secrets and set as environment variable.


#vt=20

In [ ]:
# ============================================================
# Prefix Tuning on CinePile (Phase 2)
#   - Stratified sampling on CinePile (category × hard)
#   - Only save Prefix adapter
#   - Track used sample_ids to avoid re-training on same samples
#   - Save training_log_history.csv for loss analysis
# Requirements:
#   pip install transformers datasets accelerate peft trl
# ============================================================

import os
import gc
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Set
from datetime import datetime
from collections import defaultdict

import torch
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from transformers.trainer_utils import set_seed

from peft import (
    PrefixTuningConfig,
    get_peft_model,
    TaskType,
)
from trl import SFTTrainer, SFTConfig


# ================= 1. 配置 =================

@dataclass
class PrefixConfig:
    dataset_name: str          = "tomg-group-umd/cinepile"
    train_split: str           = "train"
    test_split: str            = "test"

    max_train_samples: Optional[int] = 20000
    max_test_samples: Optional[int]  = None
    hard_oversample_ratio: float     = 1.5

    base_model_id: str         = "meta-llama/Llama-3.1-8B-Instruct"

    max_scene_length: int      = 1000
    max_seq_length: int        = 1024

    num_train_epochs: int      = 3
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float       = 5e-4   # Prefix 通常 lr 稍大
    warmup_ratio: float        = 0.05
    lr_scheduler_type: str     = "cosine"
    bf16: bool                 = True
    logging_steps: int         = 50

    output_dir: str            = "./prefix_cinepile"
    log_path: str              = "./prefix_experiment_log.csv"

    prefix_num_virtual_tokens: int = 20

    max_new_tokens: int        = 5

    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })

    seed: int                  = 42


# ================= 2. Stratified Sampler + 数据集 =================

class StratifiedSampler:
    def __init__(self, cfg: PrefixConfig):
        self.cfg = cfg

    def sample(self, data):
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(self.cfg.seed)

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {len(sampled)}")
        print("[Sampler] " + "=" * 60)
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"


class CinePileDataset:
    LETTERS = ["A", "B", "C", "D", "E"]

    def __init__(self, cfg: PrefixConfig, seen_ids: Optional[Set[int]] = None):
        self.cfg      = cfg
        self.seen_ids = seen_ids or set()
        self.train_data, self.test_data = self._load_all()
        print(f"[Dataset] Train: {len(self.train_data)} | Test: {len(self.test_data)}")

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        for i, c in enumerate(choices):
            if answer_text.strip() == c.strip():
                return cls.LETTERS[i]
        for i, c in enumerate(choices):
            if answer_text.strip() in c or c.strip() in answer_text:
                return cls.LETTERS[i]
        return "A"

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == "true"

    def _load_split(self, split: str):
        raw = load_dataset(self.cfg.dataset_name, split=split)
        samples = []
        for idx, ex in enumerate(raw):
            letter = self._answer_text_to_letter(ex["answer_key"], ex["choices"])
            samples.append({
                "sample_id":         idx,
                "movie_scene":       ex["movie_scene"],
                "question":          ex["question"],
                "choices":           ex["choices"],
                "answer_key":        letter,
                "question_category": ex["question_category"],
                "hard_split":        self._normalize_hard_split(ex["hard_split"]),
            })
        return samples

    def _load_all(self):
        train_raw = self._load_split(self.cfg.train_split)
        test_raw  = self._load_split(self.cfg.test_split)

        if self.seen_ids:
            before = len(train_raw)
            train_raw = [s for s in train_raw if s["sample_id"] not in self.seen_ids]
            print(f"[Dataset] Filtered seen samples: {before} → {len(train_raw)}")

        sampler    = StratifiedSampler(self.cfg)
        train_data = sampler.sample(train_raw)

        if self.cfg.max_test_samples:
            test_data = test_raw[:self.cfg.max_test_samples]
        else:
            test_data = test_raw

        return train_data, test_data


# ================= 3. Prompt 与评估 =================

class PromptBuilder:
    @staticmethod
    def build_train_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            f"Answer: {sample['answer_key']}"
        )

    @staticmethod
    def build_inference_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            "Answer:"
        )


class Evaluator:
    def __init__(self, cfg: PrefixConfig):
        self.cfg = cfg

    def evaluate(self, model, tokenizer, test_data: list) -> dict:
        model.eval()
        device = next(model.parameters()).device
        records = []
        for s in tqdm(test_data, desc="Eval Prefix"):
            pred = self._predict(model, tokenizer, s, device)
            records.append({
                "pred":              pred,
                "label":             s["answer_key"],
                "question_category": s["question_category"],
                "hard_split":        s["hard_split"],
            })
        df = pd.DataFrame(records)
        return self._compute_metrics(df)

    def _predict(self, model, tokenizer, sample: dict, device: torch.device) -> str:
        prompt = PromptBuilder.build_inference_prompt(sample, self.cfg.max_scene_length)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.cfg.max_seq_length,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.cfg.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        for ch in decoded:
            if ch in "ABCDE":
                return ch
        return "A"

    def _compute_metrics(self, df: pd.DataFrame) -> dict:
        correct = df["pred"] == df["label"]
        metrics = {"Overall": correct.mean(), "Overall_n": len(df)}
        for abbr, keyword in self.cfg.category_map.items():
            mask = df["question_category"].str.contains(keyword, case=False, na=False)
            n    = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f"{abbr}_n"] = n
        hard_mask = df["hard_split"] == True
        n_hard    = int(hard_mask.sum())
        metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics["Hard_n"] = n_hard
        return metrics


# ================= 4. Prefix Trainer =================

class PrefixTrainer:
    def __init__(self, cfg: PrefixConfig):
        self.cfg       = cfg
        self.model     = None
        self.tokenizer = None

    def _output_dir(self) -> str:
        os.makedirs(self.cfg.output_dir, exist_ok=True)
        return self.cfg.output_dir

    def _make_hf_dataset(self, data: list) -> Dataset:
        return Dataset.from_list([
            {"text": PromptBuilder.build_train_prompt(s, self.cfg.max_scene_length)}
            for s in data
        ])

    def _load_base_model_bf16(self):
        tokenizer = AutoTokenizer.from_pretrained(self.cfg.base_model_id)
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.padding_side = "right"
        model = AutoModelForCausalLM.from_pretrained(
            self.cfg.base_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        return model, tokenizer

    def train(self, train_data: list):
        print(f"\n[Prefix] Loading base model (bf16)...")
        model, tokenizer = self._load_base_model_bf16()

        prefix_config = PrefixTuningConfig(
            task_type          = TaskType.CAUSAL_LM,
            num_virtual_tokens = self.cfg.prefix_num_virtual_tokens,
            prefix_projection  = True,
        )
        model = get_peft_model(model, prefix_config)
        model.print_trainable_parameters()

        args = SFTConfig(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.cfg.num_train_epochs,
            per_device_train_batch_size = self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps = self.cfg.gradient_accumulation_steps,
            learning_rate               = self.cfg.learning_rate,
            warmup_ratio                = self.cfg.warmup_ratio,
            lr_scheduler_type           = self.cfg.lr_scheduler_type,
            bf16                        = self.cfg.bf16,
            logging_steps               = self.cfg.logging_steps,
            save_strategy               = "no",
            max_length                  = self.cfg.max_seq_length,
            dataset_text_field          = "text",
            report_to                   = "none",
            gradient_checkpointing      = False,
        )

        hf_dataset = self._make_hf_dataset(train_data)
        trainer = SFTTrainer(
            model            = model,
            args             = args,
            train_dataset    = hf_dataset,
            processing_class = tokenizer,
        )

        print(f"[Prefix] Training on {len(train_data)} samples...")
        trainer.train()

        log_history = trainer.state.log_history
        if len(log_history) > 0:
            df_log  = pd.DataFrame(log_history)
            log_path = os.path.join(self._output_dir(), "training_log_history.csv")
            df_log.to_csv(log_path, index=False)
            print(f"[Prefix] Step-wise log saved → {log_path}")

        trainer.save_model(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[Prefix] Saved adapter → {self._output_dir()}")

        self.model     = trainer.model
        self.tokenizer = tokenizer

    def free_memory(self):
        if self.model is not None:
            del self.model
        if self.tokenizer is not None:
            del self.tokenizer
        self.model = None
        self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()


# ================= 5. 入口（含 used_ids） =================

if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    from google.colab import drive
    drive.mount('/content/drive')

    import os
    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_Prefix"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    cfg = PrefixConfig(
        base_model_id         = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples     = 30000,
        max_test_samples      = None,
        hard_oversample_ratio = 1.5,
        num_train_epochs      = 3,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        learning_rate         = 5e-4,
        output_dir            = os.path.join(PROJECT_DIR, "prefix_cinepile"),
        log_path              = os.path.join(PROJECT_DIR, "prefix_experiment_log.csv"),
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids_prefix.txt")

    set_seed(cfg.seed)

    # 读取已用 sample_ids
    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except:
                    pass
        print(f"[UsedIDs] Prefix loaded {len(seen_ids)} seen ids from {used_ids_path}")

    dataset   = CinePileDataset(cfg, seen_ids=seen_ids)
    evaluator = Evaluator(cfg)
    trainer   = PrefixTrainer(cfg)

    trainer.train(dataset.train_data)
    metrics = evaluator.evaluate(trainer.model, trainer.tokenizer, dataset.test_data)

    print("\n[Prefix] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "prefix_metrics.csv"), index=False
    )

    # 记录本轮使用的 sample_id
    with open(used_ids_path, "a") as f:
        for s in dataset.train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] Prefix appended {len(dataset.train_data)} ids → {used_ids_path}")

    trainer.free_memory()


transformers: 5.0.0
Mounted at /content/drive
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_Prefix
[UsedIDs] Prefix loaded 30000 seen ids from /content/drive/MyDrive/266_final/phase2/CinePile_Prefix/used_sample_ids_prefix.txt


README.md: 0.00B [00:00, ?B/s]

v2/train-00000-of-00003.parquet:   0%|          | 0.00/21.9M [00:00<?, ?B/s]

v2/train-00001-of-00003.parquet:   0%|          | 0.00/23.2M [00:00<?, ?B/s]

v2/train-00002-of-00003.parquet:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

...

[Prefix] Training on 30000 samples...


Step,Training Loss
50,2.426324
100,1.732614
150,1.717052
200,1.705394
250,1.711921
300,1.678264
350,1.684254
400,1.662305
450,1.664398
500,1.664765


[Prefix] Step-wise log saved → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix/prefix_cinepile/training_log_history.csv
[Prefix] Saved adapter → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix/prefix_cinepile


Eval Prefix:   0%|          | 0/4941 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Eval Prefix: 100%|██████████| 4941/4941 [07:24<00:00, 11.10it/s]



[Prefix] ===== Eval Metrics =====
  Overall   : 61.75%  (n=4941)
  TEMP      : 45.06%  (n=688)
  CRD       : 63.64%  (n=2068)
  NPA       : 64.15%  (n=463)
  STA       : 65.73%  (n=1532)
  TH        : 63.68%  (n=190)
  Hard      : 41.49%  (n=887)
[UsedIDs] Prefix appended 30000 ids → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix/used_sample_ids_prefix.txt


In [ ]:
from datasets import load_dataset

dataset = load_dataset('tomg-group-umd/cinepile', split='test')

# 查看前20个 answer_key 的原始值
print("=== 前20个 answer_key ===")
for i in range(20):
    ak = dataset[i]['answer_key']
    print(f"  [{i}] type={type(ak).__name__}, value={repr(ak)}")

# 查看所有唯一值
print("\n=== 所有唯一 answer_key 值（完整测试集）===")
from collections import Counter
counter = Counter(str(dataset[i]['answer_key']) for i in range(len(dataset)))
for val, cnt in sorted(counter.items()):
    print(f"  {repr(val):10s} → {cnt} 条")


=== 前20个 answer_key ===
  [0] type=str, value='From anxiety to excitement'
  [1] type=str, value='Suggests next steps'
  [2] type=str, value='The rough foliage'
  [3] type=str, value='She closes her eyes and focuses on staying calm without moving at all.'
  [4] type=str, value='Rough and dense'
  [5] type=str, value='They hear an aircraft approaching.'
  [6] type=str, value='To reach a specific location'
  [7] type=str, value='They become more confident'
  [8] type=str, value='The sound of a helicopter approaching'
  [9] type=str, value='Reed'
  [10] type=str, value='The sound of a helicopter'
  [11] type=str, value='In a rocky plateau'
  [12] type=str, value='It is a valley with spots of light'
  [13] type=str, value='They use a thermal suit'
  [14] type=str, value='They are trying to remain undetected by the helicopter using thermal suits.'
  [15] type=str, value='It illuminates the surrounding foliage.'
  [16] type=str, value='Directly underneath'
  [17] type=str, value='Colleagues'

In [ ]:
# 运行这段诊断，查看实际的 question_category 值分布
from datasets import load_dataset
from collections import Counter

# dataset = load_dataset('tomg-group-umd/cinepile', split='test')

counter = Counter(dataset[i]['question_category'] for i in range(200))
print("=== question_category 实际值分布 ===")
for val, cnt in sorted(counter.items(), key=lambda x: -x[1]):
    print(f"  {cnt:4d}  {repr(val)}")


=== question_category 实际值分布 ===
    88  'Character and\nRelationship Dynamics'
    63  'Setting and\nTechnical Analysis'
    24  'Temporal'
    13  'Narrative and\nPlot Analysis'
    12  'Theme Exploration'


In [ ]:
# 快速验证（无需重新加载数据集）
test_categories = [
    'Character and\nRelationship Dynamics',
    'Setting and\nTechnical Analysis',
    'Temporal',
    'Narrative and\nPlot Analysis',
    'Theme Exploration',
]

category_map = {
    'TEMP': 'Temporal',
    'CRD':  'Character and',
    'NPA':  'Narrative and',
    'STA':  'Setting and',
    'TH':   'Theme Exploration',
}

import pandas as pd
df_test = pd.DataFrame({'question_category': test_categories})

print("=== 映射验证 ===")
for abbr, keyword in category_map.items():
    mask = df_test['question_category'].str.contains(keyword, case=False, na=False)
    matched = df_test.loc[mask, 'question_category'].tolist()
    print(f"  {abbr} ({keyword!r:20s}) → {matched}")


=== 映射验证 ===
  TEMP ('Temporal'          ) → ['Temporal']
  CRD ('Character and'     ) → ['Character and\nRelationship Dynamics']
  NPA ('Narrative and'     ) → ['Narrative and\nPlot Analysis']
  STA ('Setting and'       ) → ['Setting and\nTechnical Analysis']
  TH ('Theme Exploration' ) → ['Theme Exploration']


# Vt=50

In [ ]:


# ================= 1. 配置 =================

@dataclass
class PrefixConfig:
    dataset_name: str          = "tomg-group-umd/cinepile"
    train_split: str           = "train"
    test_split: str            = "test"

    max_train_samples: Optional[int] = 20000
    max_test_samples: Optional[int]  = None
    hard_oversample_ratio: float     = 1.5

    base_model_id: str         = "meta-llama/Llama-3.1-8B-Instruct"

    max_scene_length: int      = 1000
    max_seq_length: int        = 1024

    num_train_epochs: int      = 3
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float       = 5e-4
    warmup_ratio: float        = 0.05
    lr_scheduler_type: str     = "cosine"
    bf16: bool                 = True
    logging_steps: int         = 50

    output_dir: str            = "./prefix_cinepile"
    log_path: str              = "./prefix_experiment_log.csv"

    prefix_num_virtual_tokens: int = 50

    max_new_tokens: int        = 5

    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })

    seed: int                  = 42


# ================= 2. Stratified Sampler + DATASET =================

class StratifiedSampler:
    def __init__(self, cfg: PrefixConfig):
        self.cfg = cfg

    def sample(self, data):
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples
        groups = defaultdict(list)

        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        base_quota   = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q       = base_quota
        hard_q       = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(self.cfg.seed)

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {len(sampled)}")
        print("[Sampler] " + "=" * 60)
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"


class CinePileDataset:
    LETTERS = ["A", "B", "C", "D", "E"]

    def __init__(self, cfg: PrefixConfig, seen_ids: Optional[Set[int]] = None):
        self.cfg      = cfg
        self.seen_ids = seen_ids or set()
        self.train_data, self.test_data = self._load_all()
        print(f"[Dataset] Train: {len(self.train_data)} | Test: {len(self.test_data)}")

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        for i, c in enumerate(choices):
            if answer_text.strip() == c.strip():
                return cls.LETTERS[i]
        for i, c in enumerate(choices):
            if answer_text.strip() in c or c.strip() in answer_text:
                return cls.LETTERS[i]
        return "A"

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == "true"

    def _load_split(self, split: str):
        raw = load_dataset(self.cfg.dataset_name, split=split)
        samples = []
        for idx, ex in enumerate(raw):
            letter = self._answer_text_to_letter(ex["answer_key"], ex["choices"])
            samples.append({
                "sample_id":         idx,
                "movie_scene":       ex["movie_scene"],
                "question":          ex["question"],
                "choices":           ex["choices"],
                "answer_key":        letter,
                "question_category": ex["question_category"],
                "hard_split":        self._normalize_hard_split(ex["hard_split"]),
            })
        return samples

    def _load_all(self):
        train_raw = self._load_split(self.cfg.train_split)
        test_raw  = self._load_split(self.cfg.test_split)

        if self.seen_ids:
            before = len(train_raw)
            train_raw = [s for s in train_raw if s["sample_id"] not in self.seen_ids]
            print(f"[Dataset] Filtered seen samples: {before} → {len(train_raw)}")

        sampler    = StratifiedSampler(self.cfg)
        train_data = sampler.sample(train_raw)

        if self.cfg.max_test_samples:
            test_data = test_raw[:self.cfg.max_test_samples]
        else:
            test_data = test_raw

        return train_data, test_data


# ================= 3. Prompt and Evaluator =================

class PromptBuilder:
    @staticmethod
    def build_train_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            f"Answer: {sample['answer_key']}"
        )

    @staticmethod
    def build_inference_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Options:\n{options}\n\n"
            "Answer:"
        )


class Evaluator:
    def __init__(self, cfg: PrefixConfig):
        self.cfg = cfg

    def evaluate(self, model, tokenizer, test_data: list) -> dict:
        model.eval()
        device = next(model.parameters()).device
        records = []
        for s in tqdm(test_data, desc="Eval Prefix"):
            pred = self._predict(model, tokenizer, s, device)
            records.append({
                "pred":              pred,
                "label":             s["answer_key"],
                "question_category": s["question_category"],
                "hard_split":        s["hard_split"],
            })
        df = pd.DataFrame(records)
        return self._compute_metrics(df)

    def _predict(self, model, tokenizer, sample: dict, device: torch.device) -> str:
        prompt = PromptBuilder.build_inference_prompt(sample, self.cfg.max_scene_length)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.cfg.max_seq_length,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.cfg.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        for ch in decoded:
            if ch in "ABCDE":
                return ch
        return "A"

    def _compute_metrics(self, df: pd.DataFrame) -> dict:
        correct = df["pred"] == df["label"]
        metrics = {"Overall": correct.mean(), "Overall_n": len(df)}
        for abbr, keyword in self.cfg.category_map.items():
            mask = df["question_category"].str.contains(keyword, case=False, na=False)
            n    = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f"{abbr}_n"] = n
        hard_mask = df["hard_split"] == True
        n_hard    = int(hard_mask.sum())
        metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics["Hard_n"] = n_hard
        return metrics


# ================= 4. Prefix Trainer =================

class PrefixTrainer:
    def __init__(self, cfg: PrefixConfig):
        self.cfg       = cfg
        self.model     = None
        self.tokenizer = None

    def _output_dir(self) -> str:
        os.makedirs(self.cfg.output_dir, exist_ok=True)
        return self.cfg.output_dir

    def _make_hf_dataset(self, data: list) -> Dataset:
        return Dataset.from_list([
            {"text": PromptBuilder.build_train_prompt(s, self.cfg.max_scene_length)}
            for s in data
        ])

    def _load_base_model_bf16(self):
        tokenizer = AutoTokenizer.from_pretrained(self.cfg.base_model_id)
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.padding_side = "right"
        model = AutoModelForCausalLM.from_pretrained(
            self.cfg.base_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        return model, tokenizer

    def train(self, train_data: list):
        print(f"\n[Prefix] Loading base model (bf16)...")
        model, tokenizer = self._load_base_model_bf16()

        prefix_config = PrefixTuningConfig(
            task_type          = TaskType.CAUSAL_LM,
            num_virtual_tokens = self.cfg.prefix_num_virtual_tokens,
            prefix_projection  = True,
        )
        model = get_peft_model(model, prefix_config)
        model.print_trainable_parameters()

        args = SFTConfig(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.cfg.num_train_epochs,
            per_device_train_batch_size = self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps = self.cfg.gradient_accumulation_steps,
            learning_rate               = self.cfg.learning_rate,
            warmup_ratio                = self.cfg.warmup_ratio,
            lr_scheduler_type           = self.cfg.lr_scheduler_type,
            bf16                        = self.cfg.bf16,
            logging_steps               = self.cfg.logging_steps,
            save_strategy               = "no",
            max_length                  = self.cfg.max_seq_length,
            dataset_text_field          = "text",
            report_to                   = "none",
            gradient_checkpointing      = False,
        )

        hf_dataset = self._make_hf_dataset(train_data)
        trainer = SFTTrainer(
            model            = model,
            args             = args,
            train_dataset    = hf_dataset,
            processing_class = tokenizer,
        )

        print(f"[Prefix] Training on {len(train_data)} samples...")
        trainer.train()

        log_history = trainer.state.log_history
        if len(log_history) > 0:
            df_log  = pd.DataFrame(log_history)
            log_path = os.path.join(self._output_dir(), "training_log_history.csv")
            df_log.to_csv(log_path, index=False)
            print(f"[Prefix] Step-wise log saved → {log_path}")

        trainer.save_model(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[Prefix] Saved adapter → {self._output_dir()}")

        self.model     = trainer.model
        self.tokenizer = tokenizer

    def free_memory(self):
        if self.model is not None:
            del self.model
        if self.tokenizer is not None:
            del self.tokenizer
        self.model = None
        self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()


# ================= 5. main =================

if __name__ == "__main__":
    import transformers
    print("transformers:", transformers.__version__)

    # Removed duplicate drive.mount() call as it's already mounted.
    # from google.colab import drive
    # drive.mount('/content/drive')

    import os
    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase2/CinePile_Prefix"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    cfg = PrefixConfig(
        base_model_id         = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples     = 30000,
        max_test_samples      = None,
        hard_oversample_ratio = 1.5,
        num_train_epochs      = 3,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        learning_rate         = 5e-4,
        output_dir            = os.path.join(PROJECT_DIR, "prefix_cinepile"),
        log_path              = os.path.join(PROJECT_DIR, "prefix_experiment_log.csv"),
    )

    used_ids_path = os.path.join(PROJECT_DIR, "used_sample_ids_prefix.txt")

    set_seed(cfg.seed)

    # read sample_ids
    seen_ids: Set[int] = set()
    if os.path.exists(used_ids_path):
        with open(used_ids_path, "r") as f:
            for line in f:
                try:
                    seen_ids.add(int(line.strip()))
                except:
                    pass
        print(f"[UsedIDs] Prefix loaded {len(seen_ids)} seen ids from {used_ids_path}")

    dataset   = CinePileDataset(cfg, seen_ids=seen_ids)
    evaluator = Evaluator(cfg)
    trainer   = PrefixTrainer(cfg)

    trainer.train(dataset.train_data)
    metrics = evaluator.evaluate(trainer.model, trainer.tokenizer, dataset.test_data)

    print("\n[Prefix] ===== Eval Metrics =====")
    for k, v in metrics.items():
        if not k.endswith("_n") and v is not None:
            n = metrics.get(f"{k}_n", "")
            print(f"  {k:<10}: {v:.2%}  (n={n})")

    pd.DataFrame([metrics]).to_csv(
        os.path.join(cfg.output_dir, "prefix_metrics.csv"), index=False
    )

    # record the used sample_id
    with open(used_ids_path, "a") as f:
        for s in dataset.train_data:
            f.write(str(s["sample_id"]) + "\n")
    print(f"[UsedIDs] Prefix appended {len(dataset.train_data)} ids → {used_ids_path}")

    trainer.free_memory()


transformers: 5.0.0
Project dir: /content/drive/MyDrive/266_final/phase2/CinePile_Prefix


README.md: 0.00B [00:00, ?B/s]

v2/train-00000-of-00003.parquet:   0%|          | 0.00/21.9M [00:00<?, ?B/s]

v2/train-00001-of-00003.parquet:   0%|          | 0.00/23.2M [00:00<?, ?B/s]

v2/train-00002-of-00003.parquet:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

...

[Prefix] Training on 30000 samples...


Step,Training Loss
50,2.630076
100,1.830411
150,1.726689
200,1.704734
250,1.738218
300,1.707519
350,1.687612
400,1.682484
450,1.669421
500,1.674250


[Prefix] Step-wise log saved → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix/prefix_cinepile/training_log_history.csv
[Prefix] Saved adapter → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix/prefix_cinepile


Eval Prefix:   0%|          | 0/4941 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Eval Prefix: 100%|██████████| 4941/4941 [07:18<00:00, 11.26it/s]



[Prefix] ===== Eval Metrics =====
  Overall   : 61.85%  (n=4941)
  TEMP      : 45.93%  (n=688)
  CRD       : 64.36%  (n=2068)
  NPA       : 63.71%  (n=463)
  STA       : 65.01%  (n=1532)
  TH        : 62.11%  (n=190)
  Hard      : 41.94%  (n=887)
[UsedIDs] Prefix appended 30000 ids → /content/drive/MyDrive/266_final/phase2/CinePile_Prefix/used_sample_ids_prefix.txt
